# #248 Forward Snapshot operational notebook

Executed offline against compact synthetic Nowscore TypeID=5 payloads. It demonstrates the report shape only; it does not assert a live collection sample or historical backfill.

In [1]:
import json, sys, tempfile
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path.cwd() / 'src'))
from forward_snapshot import SnapshotStore, materialize_common_epoch_view, materialize_relative_horizon_view

def page(a, b, c, clock):
    return ('<html><head><title>Sp值变化表</title></head><body><table>'
            '<tr><td>胜</td><td>平</td><td>负</td><td>变化时间</td></tr>'
            f'<tr><td>{a}</td><td>{b}</td><td>{c}</td><td>{clock}</td></tr>'
            '</table></body></html>').encode()

root = Path(tempfile.mkdtemp(prefix='bj248-notebook-'))
store = SnapshotStore.open(root)
for match_idx, kickoff, values in [(1, '2026-05-01T04:00:00Z', (2.1,3.2,3.8)), (2, '2026-05-01T05:00:00Z', (2.2,3.1,3.7))]:
    store.record_history(capture_key=f'demo-{match_idx}', payload=page(*values, '05-01 09:00'), issue='26051', match_idx=match_idx, kickoff_at_utc=kickoff, native_handicap_line=0.0, sale_status='OPEN', issue_url='https://cp.nowscore.com/buy/DanChang.aspx?TypeID=5&issueNum=26051', requested_at_utc='2026-05-01T00:30:00Z', trusted_received_at_utc='2026-05-01T01:30:00Z', parser_version='notebook-demo', acquisition_result='OK')
epoch = {'26051': '2026-05-01T02:00:00Z'}
common = materialize_common_epoch_view(store, decision_epochs=epoch)
relative = materialize_relative_horizon_view(store, horizons_minutes=(60,))
report = {
  'issue_fixture_cell_coverage': {'issues': int(common.issue_code.nunique()), 'fixtures': int(common.fixture_key.nunique()), 'cells': int(len(common))},
  'cadence_receipts': int(store.count('capture_receipt')),
  'staleness_minutes': sorted(((pd.to_datetime(common.decision_at_utc, utc=True)-pd.to_datetime(common.source_changed_at_utc, utc=True)).dt.total_seconds()/60).unique().tolist()),
  'missingness': {'common_epoch_cells_missing': int(6-len(common)), 'relative_1h_cells_missing': int(6-len(relative))},
  'line_sp_changes': {'native_lines': sorted(common.native_handicap_line.dropna().unique().tolist()), 'distinct_sp_vectors': int(common.groupby(['fixture_key','source_changed_at_utc']).ngroups)},
  'simultaneous_visibility': common.groupby('issue_code').decision_at_utc.nunique().to_dict(),
  'feasible_N': common.groupby('issue_code').fixture_key.nunique().to_dict(),
}
print(json.dumps(report, indent=2, sort_keys=True))


{
  "cadence_receipts": 2,
  "feasible_N": {
    "26051": 2
  },
  "issue_fixture_cell_coverage": {
    "cells": 6,
    "fixtures": 2,
    "issues": 1
  },
  "line_sp_changes": {
    "distinct_sp_vectors": 2,
    "native_lines": [
      0.0
    ]
  },
  "missingness": {
    "common_epoch_cells_missing": 0,
    "relative_1h_cells_missing": 0
  },
  "simultaneous_visibility": {
    "26051": 1
  },
  "staleness_minutes": [
    60.0
  ]
}


/home/drwho1985/work/beijing-lot-206/.venv/lib/python3.11/site-packages/bs4/css.py:40: UserWarning: The soupsieve package is not installed. CSS selectors cannot be used.
  warnings.warn(
